In [6]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from dataclasses import dataclass

In [2]:
# Create our training environment - a cart with a pole that needs balancing
env = gym.make("CartPole-v1", render_mode="human")

# Reset environment to start a new episode
observation, info = env.reset()
# observation: what the agent can "see" - cart position, velocity, pole angle, etc.
# info: extra debugging information (usually not needed for basic learning)

print(f"Starting observation: {observation}")
# Example output: [ 0.01234567 -0.00987654  0.02345678  0.01456789]
# [cart_position, cart_velocity, pole_angle, pole_angular_velocity]

episode_over = False
total_reward = 0

while not episode_over:
    # Choose an action: 0 = push cart left, 1 = push cart right
    action = env.action_space.sample()  # Random action for now - real agents will be smarter!

    # Take the action and see what happens
    observation, reward, terminated, truncated, info = env.step(action)

    # reward: +1 for each step the pole stays upright
    # terminated: True if pole falls too far (agent failed)
    # truncated: True if we hit the time limit (500 steps)

    total_reward += reward
    episode_over = terminated or truncated

print(f"Episode finished! Total reward: {total_reward}")
env.close()

Starting observation: [0.01469224 0.02984197 0.01033643 0.04968922]
Episode finished! Total reward: 36.0


In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class CustomCarEnv(gym.Env):
    metadata = {"render_modes": ["human"], "render_fps": 30}

    def __init__(self, render_mode=None):
        super().__init__()
        
        # Azioni: ad esempio accelerazione continua [-1.0, 1.0]
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)
        
        # Osservazioni: [posizione, velocità]
        self.observation_space = spaces.Box(
            low=np.array([-5.0, -2.0], dtype=np.float32),
            high=np.array([5.0, 2.0], dtype=np.float32),
            shape=(2,),
            dtype=np.float32
        )
        
        self.render_mode = render_mode
        self.state = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        # Stato iniziale casuale o fisso
        self.state = np.array([0.0, 0.0], dtype=np.float32)
        info = {}
        
        return self.state, info

    def step(self, action):
        position, velocity = self.state
        acceleration = np.clip(action[0], self.action_space.low[0], self.action_space.high[0])
        
        # Aggiornamento fisico semplice
        velocity += acceleration * 0.1
        position += velocity
        
        self.state = np.array([position, velocity], dtype=np.float32)
        
        # Condizioni di terminazione e ricompensa
        terminated = bool(position >= 4.0)
        truncated = bool(position <= -4.0)
        
        reward = 1.0 if terminated else -0.1
        info = {}
        
        return self.state, reward, terminated, truncated, info

    def render(self):
        if self.render_mode == "human":
            print(f"Posizione: {self.state[0]:.2f}, Velocità: {self.state[1]:.2f}")

from gymnasium.envs.registration import register

register(
    id="CustomCar-v0",
    entry_point="__main__:CustomCarEnv", # Oppure "mio_modulo.envs:CustomCarEnv"
)
env = gym.make("CustomCar-v0", render_mode="human")
observation, info = env.reset(seed=42)

episode_over = False
total_reward = 0

while not episode_over:
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)

KeyboardInterrupt: 

In [8]:
class Config:
    # Boat configuration parameters
    max_speed: float = 10.0
    sail_rotation_speed: float = 0.1
    boat_rotation_speed: float = 0.05
    initial_sail_angle: float = 0.0
    initial_boat_angle: float = 0.0
    initial_position: np.ndarray = np.array([0.0, 0.0])

    # Map configuration parameters
    map_width: int = 100
    map_height: int = 100
    water_friction: float = 0.1

    # Simulation parameters
    dt: float = 0.1
    max_steps: int = 200
    

In [ ]:
class Checkpoint:
    def __init__(self, position, number, radius):
        self.position = position
        self.number = number
        self.visited = False
        self.radius = radius


class VecField():
    def __init__(self, space_length, space_width, function):
        pass
    def get_vec(self, point2d):
        return

class SailingEnv(gym.Env):

    def __init__(self, config: Config, wind_vec_field: VecField, goal: Checkpoint, checkpoints: list):
        self.checkpoints = checkpoints
        self.n_checkpoints = len(checkpoints)
        self.goal = goal
        
        self.state = None
        self.steps = 0
        self.max_steps = config.max_steps
        self.max_speed = config.max_speed
        self.friction_coefficient = config.water_friction

        
        
        #ACTIONS = ROTATE_LEFT_SAIL, ROTATE_RIGHT_SAIL, ROTATE_LEFT_BOAT, ROTATE_RIGHT_BOAT
        # The action space is a continuous 2D vector representing the rotation angle of the sail and the boat
        # for simplicity it will be parameterized as a 2D vector with values in the range [-1, 1] for both dimensions
        self.action_space = spaces.Dict({
            "sail_rotation": spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float),
            "boat_rotation": spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float)
        })

        # OBSERVATIONS = BOAT_POSITION, BOAT_VELOCITY, SAIL_ANGLE, WIND_VECTOR, CP_relative_positions and goal_relative_position
        self.observation_space = spaces.Dict({
            "boat_position": spaces.Box(low=np.array([0, 0]), high=np.array([config.map_width, config.map_height]), dtype=np.float),
            "boat_velocity": spaces.Box(low=np.array([-np.inf, -np.inf]), high=np.array([np.inf, np.inf]), dtype=np.float),
            "sail_angle": spaces.Box(low=-np.pi, high=np.pi, dtype=np.float),
            "boat_angle": spaces.Box(low=np.pi, high=np.pi, dtype=np.float),
            "wind_vector": spaces.Box(low=np.array([-np.inf, -np.inf]), high=np.array([np.inf, np.inf]), dtype=np.float),
            "checkpoints_relatives": spaces.MultiBinary(self.n_checkpoints),
            "goal_position": spaces.Box(low=np.array([0, 0]), high=np.array([config.map_width, config.map_height]), dtype=np.float)
        })

        
        self.initial_state = {"boat_position": config.initial_position,
                              "boat_velocity": np.array([0., 0.]),
                              "sail_angle": 0.,
                              "boat_angle": config.initial_boat_angle,
                              "wind_vector": wind_vec_field.get_vec(config.initial_position),
                              "checkpoints_relatives": [self._calc_relative_dist_(cp) for cp in self.checkpoints],
                              "goal_relatives": self._calc_relative_dist_(self.goal)}
        

    def _calc_relative_dist_(self, point: Checkpoint):
        boat_x, boat_y = self.state["boat_position"]
        point_x, point_y = point.position
    
        dx = point_x - boat_x
        dy = point_y - boat_y
        distance = np.linalg.norm(self.state["boat_position"] - point.position)

        return np.array([dx, dy, distance])

    
    def reset(self, seed=None):
        super().reset(seed=seed)
        
        self.state = self.initial_state.copy()
        return self.state, info

    def step(self, action):

        state = self.state
        new_sail_angle = state["sail_angle"] + action["sail_rotation"][0] * self.sail_rotation_speed * self.dt
        new_boat_angle = state["boat_angle"] + action["boat_rotation"][0] * self.boat_rotation_speed * self.dt
        self.state["sail_angle"] = np.clip(new_sail_angle, -np.pi, np.pi)
        self.state["boat_angle"] = np.clip(new_boat_angle, -np.pi, np.pi)


        wind_force = self.calculate_wind_force()
        
        velocity +=  wind_force * self.dt
        position += velocity

        reward = self.reward_function()
        

        self.steps += 1

        if self.steps >= self.max_steps:
            truncated = True
            terminated = True

        info = {}
        return state, reward, terminated, truncated, info
    
    def calculate_velocity(self, wind_force, boat_angle):
        # Placeholder
        velocity = 12
        return velocity
    
    def calculate_wind_force(self):
        # Placeholder
        wind_velocity = self.state["wind_vector"]
        sail_angle = self.state["sail_angle"]
        # Calculate the wind force based on the wind velocity and sail angle
        wind_force = np.array([wind_velocity[0] * np.cos(sail_angle), wind_velocity[1] * np.sin(sail_angle)])
        return wind_force
    
    def reward_function(self):
        pass
        

